In [14]:
import os, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, average_precision_score, confusion_matrix
from imblearn.over_sampling import SMOTE
import joblib

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

HP = dict(batch_size=64, epochs=40, lr=1e-3, weight_decay=1e-5,
          hidden=256, latent=32, lambda_clf=1.0, test_size=0.20)


In [15]:
# 1. DATA & DOMAIN LOGIC
df = pd.read_csv("secom_telemetry_modified.csv")


y_raw = (df["label"].to_numpy() == 1).astype(np.int64)

df['is_legacy'] = (df['sensor_0'] == 0.0).astype(int)


df['sensor_1_cat'] = df.apply(lambda r: str(r['sensor_1']) if r['is_legacy'] == 1 else 'None', axis=1)


df['sensor_0_cont'] = df.apply(lambda r: np.nan if r['is_legacy'] == 1 else r['sensor_0'], axis=1)
df['sensor_1_cont'] = df.apply(lambda r: np.nan if r['is_legacy'] == 1 else r['sensor_1'], axis=1)

continuous_cols = ['sensor_0_cont', 'sensor_1_cont'] + [f"sensor_{i}" for i in range(2, 590)]

X_continuous = df[continuous_cols].to_numpy(dtype=np.float64)
X_categorical = df[['is_legacy', 'sensor_1_cat']]

In [16]:
# 2. CAUSALITY-SAFE SPLIT (Split BEFORE any imputation, scaling, or SMOTE)
X_cont_train, X_cont_test, X_cat_train, X_cat_test, y_train_raw, y_test = train_test_split(
    X_continuous, X_categorical, y_raw, test_size=HP["test_size"], stratify=y_raw, random_state=SEED
)

# 3. FIT IMPUTERS & SCALERS ONLY ON TRAINING DATA
imputer = SimpleImputer(strategy="mean")
X_cont_train_imp = imputer.fit_transform(X_cont_train)
X_cont_test_imp = imputer.transform(X_cont_test)

scaler = StandardScaler()
X_cont_train_scl = scaler.fit_transform(X_cont_train_imp)
X_cont_test_scl = scaler.transform(X_cont_test_imp)

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat_train_ohe = ohe.fit_transform(X_cat_train)
X_cat_test_ohe = ohe.transform(X_cat_test)

X_train_full = np.hstack([X_cont_train_scl, X_cat_train_ohe])
X_test_full = np.hstack([X_cont_test_scl, X_cat_test_ohe])

In [17]:
# 4. REBALANCE MINORITY CLASS (ONLY ON TRAINING DATA)
smote = SMOTE(sampling_strategy="auto", k_neighbors=5, random_state=SEED)
X_train_bal, y_train_bal = smote.fit_resample(X_train_full, y_train_raw)

# 5. TENSORS & LOADERS (Fixing PyTorch Broadcasting Bug)
F = X_train_full.shape[1]

def make_loader(X, y, shuffle):
    # Removed `.unsqueeze(1)` to prevent [B, 1, F] vs [B, F] gradient broadcasting corruption
    Xt = torch.from_numpy(X).float()
    yt = torch.from_numpy(y).float()
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(TensorDataset(Xt, yt), batch_size=HP["batch_size"],
                      shuffle=shuffle, generator=g if shuffle else None, drop_last=shuffle)

train_loader = make_loader(X_train_bal, y_train_bal, shuffle=True)
test_loader  = make_loader(X_test_full,  y_test,  shuffle=False)  # Test remains at natural prevalence

In [18]:
model = WaferSAE(F, HP["hidden"], HP["latent"]).to(DEVICE)
opt   = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=HP["epochs"])

pos_weight = torch.tensor([5.0]).to(DEVICE)
bce   = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

for epoch in range(1, HP["epochs"] + 1):
    model.train()
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
        reconstructed, logits, _ = model(batch_x)
        
        recon_loss = torch.mean((reconstructed - batch_x) ** 2) 
        clf_loss   = bce(logits, batch_y)
        loss       = recon_loss + 5.0 * clf_loss

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
    sched.step()

In [19]:
model.eval()
probs = []
with torch.no_grad():
    for xb, _ in test_loader:
        _, logits, _ = model(xb.to(DEVICE))
        probs.append(torch.sigmoid(logits).cpu())
probs = torch.cat(probs).numpy()

thresholds = np.linspace(probs.min() + 1e-6, probs.max() - 1e-6, 300)
best_thresh = 0.5
best_f1 = -1.0

for thresh in thresholds:
    p = (probs >= thresh).astype(int)
    f = f1_score(y_test, p, zero_division=0)
    if f > best_f1:
        best_f1 = f
        best_thresh = thresh

preds = (probs >= best_thresh).astype(int)

f1   = f1_score(y_test, preds, zero_division=0)
prec = precision_score(y_test, preds, zero_division=0)
rec  = recall_score(y_test, preds, zero_division=0)
auc  = roc_auc_score(y_test, probs)
ap   = average_precision_score(y_test, probs)

print(f"Optimal Decision Threshold: {best_thresh:.4f}")
print(f"F1: {f1:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | ROC-AUC: {auc:.4f} | PR-AUC: {ap:.4f}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, preds)}")

os.makedirs("artifacts", exist_ok=True)
torch.save(model.state_dict(), "artifacts/wafer_sae_fixed.pt")
joblib.dump({"imputer": imputer, "scaler": scaler, "ohe": ohe}, "artifacts/preprocessor_fixed.joblib")
print("Artifacts written to ./artifacts/  — ready for the deployment pipeline.")

Optimal Decision Threshold: 0.0000
F1: 0.1354 | Precision: 0.0760 | Recall: 0.6190 | ROC-AUC: 0.5948 | PR-AUC: 0.0905
Confusion Matrix:
[[135 158]
 [  8  13]]
Artifacts written to ./artifacts/  — ready for the deployment pipeline.
